<a href="https://colab.research.google.com/github/johncase17/Data-Science-Project-1/blob/main/KPConv_5Class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

NOTEBOOK_DIR  = '/content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL'
DRIVE_RESULTS = f'{NOTEBOOK_DIR}/results/kpconv'
print(f'Notebook dir: {NOTEBOOK_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook dir: /content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL


In [2]:
# ⚠️ Runtime restarts after this cell. Re-run from Cell 0 after restart.
import os; os.chdir('/')
!pip install -q condacolab
import condacolab
condacolab.install()


✨🍰✨ Everything looks OK!


In [3]:
%%bash
cd /
conda create -y -n kpconv python=3.8
conda env list


Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local/envs/kpconv

  added / updated specs:
    - python=3.8


The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-20_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-hda65f42_9 
  ca-certificates    conda-forge/noarch::ca-certificates-2026.6.17-hbd8a1cb_0 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.45.1-default_hbd61a6d_102 
  libffi             conda-forge/linux-64::libffi-3.7.0-h3435931_0 
  libgcc             conda-forge/linux-64::libgcc-15.2.0-he0feb66_19 
  libgcc-ng          conda-forge/linux-64::libgcc-ng-15.2.0-h69a702a_19 
  libgomp            conda-forge/linux-64::libgomp-15.2.0-he0feb66_19 
  liblzma            conda-forge/linux-64::liblzma-5.8.3-hb03c661_0 
  liblzma-devel      conda-forge/linux-64::liblzma-devel-5.8.3-hb03c661_0 
  libnsl             conda



==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda




In [4]:
%%bash
cd /
/usr/local/envs/kpconv/bin/pip install --quiet \
    numpy==1.21.6 scipy scikit-learn matplotlib pyyaml setuptools==59.5.0

/usr/local/envs/kpconv/bin/pip install --quiet \
    torch==1.10.0+cu113 torchvision==0.11.1+cu113 \
    --extra-index-url https://download.pytorch.org/whl/cu113

echo 'Dependencies installed.'


Dependencies installed.


In [5]:
%%bash
cd /
if [ ! -d /content/KPConv-PyTorch ]; then
    git clone https://github.com/HuguesTHOMAS/KPConv-PyTorch.git /content/KPConv-PyTorch
fi
echo 'Repo contents:'
ls /content/KPConv-PyTorch


Repo contents:
cpp_wrappers
datasets
doc
INSTALL.md
kernels
LICENSE.txt
models
plot_convergence.py
README.md
test_models.py
train_ModelNet40.py
train_NPM3D.py
train_S3DIS.py
train_SemanticKitti.py
train_SensatUrban.py
train_Toronto3D.py
utils
visualize_deformations.py


Cloning into '/content/KPConv-PyTorch'...


In [6]:
import zipfile, time, os

NOTEBOOK_DIR  = '/content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL'
DRIVE_RESULTS = f'{NOTEBOOK_DIR}/results/kpconv'

ZIP_PATH = os.path.join(NOTEBOOK_DIR, 'semkitti_research.zip')
DATA_DIR = '/content/KPConv-PyTorch/Data/SemanticKitti'
SEQ_DST  = os.path.join(DATA_DIR, 'sequences')
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(SEQ_DST):
    print('Extracting sequences from zip...')
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        members = [m for m in z.namelist()
                   if 'semantickitti_subsampled/sequences/' in m]
        for member in members:
            rel = member.split('semantickitti_subsampled/sequences/')[-1]
            if not rel: continue
            dst = os.path.join(SEQ_DST, rel)
            if member.endswith('/'):
                os.makedirs(dst, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(member) as src, open(dst, 'wb') as out:
                    out.write(src.read())
    print(f'Done in {time.time()-t0:.0f}s')
else:
    print('Already extracted, skipping.')

# Download semantic-kitti.yaml
YAML_DST = os.path.join(DATA_DIR, 'semantic-kitti.yaml')
if not os.path.exists(YAML_DST):
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PRBonn/semantic-kitti-api/master/config/semantic-kitti.yaml',
        YAML_DST)
    print('semantic-kitti.yaml downloaded.')

# Patch SemanticKitti.py — absolute path
sk_path = '/content/KPConv-PyTorch/datasets/SemanticKitti.py'
with open(sk_path) as f:
    src = f.read()
src = src.replace("self.path = '../../Data/SemanticKitti'",
                  "self.path = '/content/KPConv-PyTorch/Data/SemanticKitti'")
with open(sk_path, 'w') as f:
    f.write(src)

# Patch trainer.py — allow_pickle
trainer_path = '/content/KPConv-PyTorch/utils/trainer.py'
with open(trainer_path) as f:
    src = f.read()
src = src.replace('np.load(filepath)', 'np.load(filepath, allow_pickle=True)')
with open(trainer_path, 'w') as f:
    f.write(src)

print(f'Sequences: {sorted(os.listdir(SEQ_DST))}')
print('Patches applied.')


Extracting sequences from zip...
Done in 97s
semantic-kitti.yaml downloaded.
Sequences: ['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21']
Patches applied.


In [7]:
%%bash
cd /content/KPConv-PyTorch/cpp_wrappers
export PATH=/usr/local/envs/kpconv/bin:$PATH

echo '--- Python being used ---'
which python && python --version

echo '--- Compiling wrappers ---'
bash compile_wrappers.sh && echo 'SUCCESS: C++ wrappers compiled.' || echo 'FAILED'


--- Python being used ---
/usr/local/envs/kpconv/bin/python
Python 3.8.20
--- Compiling wrappers ---
running build_ext
building 'grid_subsampling' extension
Make sure that Python modules winreg, win32api or win32con are installed.
C compiler: gcc -pthread -B /usr/local/envs/kpconv/compiler_compat -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /usr/local/envs/kpconv/include -fPIC -O2 -isystem /usr/local/envs/kpconv/include -fPIC

creating build
creating build/temp.linux-x86_64-3.8
creating build/temp.linux-x86_64-3.8/cpp_wrappers
creating build/temp.linux-x86_64-3.8/cpp_wrappers/cpp_utils
creating build/temp.linux-x86_64-3.8/cpp_wrappers/cpp_utils/cloud
creating build/temp.linux-x86_64-3.8/grid_subsampling
compile options: '-I/usr/local/envs/kpconv/lib/python3.8/site-packages/numpy/core/include -I/usr/local/envs/kpconv/include/python3.8 -c'
extra options: '-std=c++11 -D_GLIBCXX_USE_CXX11_ABI=0'
gcc: ../cpp_utils/cloud/cloud.cpp
gcc: wrapper.cpp
gcc: grid

In file included from /usr/local/envs/kpconv/lib/python3.8/site-packages/numpy/core/include/numpy/ndarraytypes.h:1969,
                 from /usr/local/envs/kpconv/lib/python3.8/site-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/envs/kpconv/lib/python3.8/site-packages/numpy/core/include/numpy/arrayobject.h:4,
                 from wrapper.cpp:2:
/usr/local/envs/kpconv/lib/python3.8/site-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~
grid_subsampling/grid_subsampling.cpp: In function ‘void grid_subsampling(std::vector<PointXYZ>&, std::vector<PointXYZ>&, std::vector<float>&, std::vector<float>&, std::vector<int>&, std::vector<int>&, float, int)’:
grid_subsampling/grid_subsampling.cpp:99:39: warning: comparison of integer expr

In [32]:
# ── Per-class IoU table ───────────────────────────────────────────
unet_per_seed   = np.array([unet_dfs[seed].loc[unet_dfs[seed]['val_miou'].idxmax(),
                             [c for c in unet_dfs[seed].columns if c.startswith('val_iou_')]].values
                             for seed in SEEDS])
hybrid_per_seed = np.array([hybrid_dfs[seed].loc[hybrid_dfs[seed]['val_miou'].idxmax(),
                             [c for c in hybrid_dfs[seed].columns if c.startswith('val_iou_')]].values
                             for seed in SEEDS])
kpconv_per_seed = np.array([kpconv_5class[seed] for seed in SEEDS])

unet_mean,   unet_std   = unet_per_seed.mean(0),   unet_per_seed.std(0)
hybrid_mean, hybrid_std = hybrid_per_seed.mean(0), hybrid_per_seed.std(0)
kpconv_mean, kpconv_std = kpconv_per_seed.mean(0), kpconv_per_seed.std(0)

rows = []
for c, name in enumerate(TERRAIN_CLASSES):
    rows.append({
        'Class':   name,
        'U-Net':   f'{unet_mean[c]:.4f} ± {unet_std[c]:.4f}',
        'Hybrid':  f'{hybrid_mean[c]:.4f} ± {hybrid_std[c]:.4f}',
        'KPConv':  f'{kpconv_mean[c]:.4f} ± {kpconv_std[c]:.4f}',
    })

meaningful = [1, 2, 3]
rows.append({
    'Class':  'mIoU',
    'U-Net':  f'{unet_mean.mean():.4f} ± {unet_per_seed.mean(1).std():.4f}',
    'Hybrid': f'{hybrid_mean.mean():.4f} ± {hybrid_per_seed.mean(1).std():.4f}',
    'KPConv': f'{kpconv_mean.mean():.4f} ± {kpconv_per_seed.mean(1).std():.4f}',
})
rows.append({
    'Class':  'mIoU (excl. Unknown+Void)',
    'U-Net':  f'{unet_mean[meaningful].mean():.4f} ± {unet_per_seed[:,meaningful].mean(1).std():.4f}',
    'Hybrid': f'{hybrid_mean[meaningful].mean():.4f} ± {hybrid_per_seed[:,meaningful].mean(1).std():.4f}',
    'KPConv': f'{kpconv_mean[meaningful].mean():.4f} ± {kpconv_per_seed[:,meaningful].mean(1).std():.4f}',
})

df_class = pd.DataFrame(rows)
print(df_class.to_string(index=False))
df_class.to_csv(os.path.join(NOTEBOOK_DIR, 'results', 'per_class_table.csv'), index=False)
print('\nSaved to per_class_table.csv')

NameError: name 'unet_dfs' is not defined

In [27]:
!grep -n "epoch_steps" /content/KPConv-PyTorch/train_SemanticKitti.py

163:    epoch_steps = 500


In [29]:
!/usr/local/envs/kpconv/bin/python /content/run_kpconv_eval.py 42 "/content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL" "/content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL/results/kpconv"


Seed 42: loading /content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL/results/kpconv/42/checkpoints/chkp_0470.tar
validation_size: 200
/usr/local/envs/kpconv/lib/python3.8/site-packages/torch/utils/data/dataloader.py:478: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(

Starting Calibration of max_in_points value (use verbose=True for more details)
Calibration done in 0.0s


Starting Calibration (use verbose=True for more details)
Calibration done in 0.0s

Loaded. Epoch: 469
  Processed 0/200 batches
  Processed 50/200 batches
  Processed 100/200 batches
  Processed 150/200 batches
Seed 42 — 5-class mIoU: 0.5420
  Unknown         

In [30]:
!/usr/local/envs/kpconv/bin/python /content/run_kpconv_eval.py 123 "/content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL" "/content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL/results/kpconv"


Seed 123: loading /content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL/results/kpconv/123/checkpoints/chkp_0600.tar
validation_size: 200
/usr/local/envs/kpconv/lib/python3.8/site-packages/torch/utils/data/dataloader.py:478: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(

Starting Calibration of max_in_points value (use verbose=True for more details)
Calibration done in 0.0s


Starting Calibration (use verbose=True for more details)
Calibration done in 0.0s

Loaded. Epoch: 599
  Processed 0/200 batches
  Processed 50/200 batches
  Processed 100/200 batches
  Processed 150/200 batches
Seed 123 — 5-class mIoU: 0.5310
  Unknown      

In [31]:
!/usr/local/envs/kpconv/bin/python /content/run_kpconv_eval.py 456 "/content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL" "/content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL/results/kpconv"


Seed 456: loading /content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL/results/kpconv/456/checkpoints/chkp_0605.tar
validation_size: 200
/usr/local/envs/kpconv/lib/python3.8/site-packages/torch/utils/data/dataloader.py:478: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(

Starting Calibration of max_in_points value (use verbose=True for more details)
Calibration done in 0.0s


Starting Calibration (use verbose=True for more details)
Calibration done in 0.0s

Loaded. Epoch: 604
  Processed 0/200 batches
  Processed 50/200 batches
  Processed 100/200 batches
  Processed 150/200 batches
Seed 456 — 5-class mIoU: 0.5322
  Unknown      

In [18]:
import numpy as np, os

NOTEBOOK_DIR  = '/content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL'
DRIVE_RESULTS = f'{NOTEBOOK_DIR}/results/kpconv'
TERRAIN_CLASSES = ['Unknown','Flat_Traversable','Slope','Obstacle','Void']
SEEDS = [42, 123, 456]

all_ious = []
for seed in SEEDS:
    iou_path = os.path.join(DRIVE_RESULTS, str(seed), 'val_preds_best', 'iou_5class.npy')
    if os.path.exists(iou_path):
        iou = np.load(iou_path)
        miou = iou.mean()
        all_ious.append(iou)
        print(f'Seed {seed} — mIoU: {miou:.4f}')
        for c, name in enumerate(TERRAIN_CLASSES):
            print(f'  {name:<22}: {iou[c]:.4f}')
    else:
        print(f'Seed {seed}: results not found at {iou_path}')

if all_ious:
    all_ious = np.array(all_ious)
    mean_iou = all_ious.mean(0)
    std_iou  = all_ious.std(0)
    print(f'\nMean mIoU: {mean_iou.mean():.4f} ± {all_ious.mean(1).std():.4f}')
    for c, name in enumerate(TERRAIN_CLASSES):
        print(f'  {name:<22}: {mean_iou[c]:.4f} ± {std_iou[c]:.4f}')

Seed 42 — mIoU: 0.0552
  Unknown               : 0.0469
  Flat_Traversable      : 0.0728
  Slope                 : 0.0791
  Obstacle              : 0.0769
  Void                  : 0.0000
Seed 123 — mIoU: 0.0550
  Unknown               : 0.0468
  Flat_Traversable      : 0.0761
  Slope                 : 0.0798
  Obstacle              : 0.0721
  Void                  : 0.0000
Seed 456 — mIoU: 0.0556
  Unknown               : 0.0472
  Flat_Traversable      : 0.0758
  Slope                 : 0.0825
  Obstacle              : 0.0726
  Void                  : 0.0000

Mean mIoU: 0.0552 ± 0.0003
  Unknown               : 0.0470 ± 0.0002
  Flat_Traversable      : 0.0749 ± 0.0015
  Slope                 : 0.0805 ± 0.0015
  Obstacle              : 0.0739 ± 0.0022
  Void                  : 0.0000 ± 0.0000


In [17]:
import os
KPCONV_DIR = '/content/drive/MyDrive/Colab Notebooks/TerrainClassificationDL/results/kpconv'
for seed in [42, 123, 456]:
    d = os.path.join(KPCONV_DIR, str(seed), 'val_preds_best')
    print(f'Seed {seed}: {os.listdir(d) if os.path.exists(d) else "NOT FOUND"}')


Seed 42: ['predictions', 'val_preds', 'parameters.txt', 'subpart_IoUs.txt', 'val_IoUs.txt', 'iou_5class.npy', 'miou_5class.npy']
Seed 123: ['predictions', 'val_preds', 'parameters.txt', 'val_IoUs.txt', 'subpart_IoUs.txt', 'iou_5class.npy', 'miou_5class.npy']
Seed 456: ['predictions', 'val_preds', 'parameters.txt', 'subpart_IoUs.txt', 'val_IoUs.txt', 'iou_5class.npy', 'miou_5class.npy']
